In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
from memoria_icm.datos import *
from memoria_icm.funciones import *
from memoria_icm.graficos import *

Este cuadernillo analiza la existencia de diferencias significativas entre los grupos del mismo nivel de salinidad, agrupados por especie. RECUPERACIÓN.

In [10]:
df, df_resumen = cargar_datos() #estos son los definitivos


columnas_ccr = [
    'r_centrado_promedio',
    'r_centrado_mediana',
    'dr_centrado_promedio',
    'dr_centrado_mediana'
]

columnas_lar = [
    'a_promedio',
    'a_mediana',
    'dr_centrado_promedio',
    'dr_centrado_mediana',
    'a_angular_centrado_mediana',
    'theta_v_promedio'
]

Datos cargados. Trackeo: (44324, 28), Resumen: (192, 42)


In [11]:
#grupos
cc_baja = df_resumen[
    (df_resumen['nivel_salinidad'] == 'baja') &
    (df_resumen['especie'] == 'CC') &
    (df_resumen['tratamiento'] == 'R')]

cc_normal = df_resumen[
    (df_resumen['nivel_salinidad'] == 'normal') &
    (df_resumen['especie'] == 'CC') &
    (df_resumen['tratamiento'] == 'R')]

cc_alta = df_resumen[
    (df_resumen['nivel_salinidad'] == 'alta') &
    (df_resumen['especie'] == 'CC') &
    (df_resumen['tratamiento'] == 'R')]

la_baja = df_resumen[
    (df_resumen['nivel_salinidad'] == 'baja') &
    (df_resumen['especie'] == 'LA') &
    (df_resumen['tratamiento'] == 'R')]

la_normal = df_resumen[
    (df_resumen['nivel_salinidad'] == 'normal') &
    (df_resumen['especie'] == 'LA') &
    (df_resumen['tratamiento'] == 'R')]

la_alta = df_resumen[
    (df_resumen['nivel_salinidad'] == 'alta') &
    (df_resumen['especie'] == 'LA') &
    (df_resumen['tratamiento'] == 'R')]

In [12]:
locos = {'cc_baja': cc_baja, 'cc_normal': cc_normal, 'cc_alta': cc_alta}
erizos = {'la_baja': la_baja, 'la_normal': la_normal, 'la_alta': la_alta}
resultados_cc = {}
resultados_la = {}
factor = 'salinidad'

#cc
for nombre_grupo, df in locos.items():
    clave = f"{nombre_grupo}"
    resultados_cc[clave] = pd.concat([testear_supuestos(df, var, factor) for var in columnas_ccr],
                                  ignore_index=True)
    
df_resultados_cc = pd.concat([pd.DataFrame(lista).assign(grupo=clave) for clave, lista in resultados_cc.items()],
                          ignore_index=True)

#la
for nombre_grupo, df in erizos.items():
    clave = f"{nombre_grupo}"
    resultados_la[clave] = pd.concat([testear_supuestos(df, var, factor) for var in columnas_lar],
                                  ignore_index=True)
    
df_resultados_la = pd.concat([pd.DataFrame(lista).assign(grupo=clave) for clave, lista in resultados_la.items()],
                          ignore_index=True)

#este dataframe entrega resultados de TODOS LOS GRUPOS y TODAS LAS variables_resumen_resumen de una sola vez,
#luego filtro

Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores
Shapiro requiere al menos 3 valores


c:\Users\franc\anaconda3\envs\tesis\Lib\site-packages\numpy\_core\fromnumeric.py:4268: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\franc\anaconda3\envs\tesis\Lib\site-packages\numpy\_core\_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\franc\anaconda3\envs\tesis\Lib\site-packages\scipy\stats\_morestats.py:2903: RuntimeWarning: divide by zero encountered in divide
  * ((xp.sum(1/(Ni - 1), axis=0, dtype=dtype)) - 1/(Ntot - k)))
c:\Users\franc\anaconda3\envs\tesis\Lib\site-packages\numpy\_core\fromnumeric.py:4268: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\franc\anaconda3\envs\tesis\Lib\site-packages\numpy\_core\_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\franc\anaconda3\

# LOCOS

## baja

### Análisis de la varianza

In [13]:
resultados_cc_baja = df_resultados_cc[(df_resultados_cc['grupo'] == 'cc_baja')]

In [14]:
variables_resumen_por_test = (resultados_cc_baja[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("LOCOS BAJA - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_cc_baja.dropna(subset=['test_sugerido']).groupby('test_sugerido')}

LOCOS BAJA - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['r_centrado_promedio', 'r_centrado_mediana', 'dr_centrado_promedio', 'dr_centrado_mediana']


In [15]:
df_efectos_cc_baja = analizar_grupo(cc_baja, listas_por_test, factor)

pvalues_significativos_cc_baja = df_efectos_cc_baja.loc[(df_efectos_cc_baja['p_value'] < 0.05)]
variables_candidatas_cc_baja = pvalues_significativos_cc_baja['variable'].tolist()
pvalues_significativos_cc_baja

,variable,test,p_value,η²,ω²/ε²


### Pruebas post hoc

In [16]:
variables_tukey = list(df_efectos_cc_baja.loc[
    (df_efectos_cc_baja['test'] == 'ANOVA') &
    (df_efectos_cc_baja['variable'].isin(variables_candidatas_cc_baja)),
    'variable'])

variables_gh = list(df_efectos_cc_baja.loc[
    (df_efectos_cc_baja['test'] == "Welch's ANOVA") &
    (df_efectos_cc_baja['variable'].isin(variables_candidatas_cc_baja)),
    'variable'])

variables_dunn = list(df_efectos_cc_baja.loc[
    (df_efectos_cc_baja['test'] == 'Kruskal-Wallis') &
    (df_efectos_cc_baja['variable'].isin(variables_candidatas_cc_baja)),
    'variable'])

In [9]:
if variables_tukey:
    tukey_df = posthoc_anova_tukey(cc_baja, variables_tukey, factor)
    
if variables_gh:
    gh_df = posthoc_welch_gameshowell(cc_baja, variables_gh, factor)
    
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(cc_baja, variables_dunn, factor)

In [10]:
if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
    
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
    
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")

In [11]:
# print(posthocs_tukey['a_angular_promedio'])

## normal

In [18]:
resultados_cc_normal = df_resultados_cc[(df_resultados_cc['grupo'] == 'cc_normal')]
variables_resumen_por_test = (resultados_cc_normal[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("LOCOS NORMAL - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_cc_normal.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_cc_normal = analizar_grupo(cc_normal, listas_por_test, factor)

pvalues_significativos_cc_normal = df_efectos_cc_normal.loc[(df_efectos_cc_normal['p_value'] < 0.05)]
variables_candidatas_cc_normal = pvalues_significativos_cc_normal['variable'].tolist()
#variables_candidatas_cc_normal #podemos mostrar el dataframe completo también
df_efectos_cc_normal


LOCOS NORMAL - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar Welch's ANOVA en: ['r_centrado_promedio', 'r_centrado_mediana', 'dr_centrado_promedio', 'dr_centrado_mediana']


,variable,test,p_value,η²,ω²/ε²
2,dr_centrado_promedio,Welch's ANOVA,0.043381,0.407234,0.293921
0,r_centrado_promedio,Welch's ANOVA,0.056915,0.379792,0.262839
1,r_centrado_mediana,Welch's ANOVA,0.064292,0.367064,0.248469
3,dr_centrado_mediana,Welch's ANOVA,0.177738,0.250169,0.117834


In [7]:
### Pruebas post hoc
variables_tukey = list(df_efectos_cc_normal.loc[
    (df_efectos_cc_normal['test'] == 'ANOVA') &
    (df_efectos_cc_normal['variable'].isin(variables_candidatas_cc_normal)),
    'variable'])

variables_gh = list(df_efectos_cc_normal.loc[
    (df_efectos_cc_normal['test'] == "Welch's ANOVA") &
    (df_efectos_cc_normal['variable'].isin(variables_candidatas_cc_normal)),
    'variable'])

variables_dunn = list(df_efectos_cc_normal.loc[
    (df_efectos_cc_normal['test'] == 'Kruskal-Wallis') &
    (df_efectos_cc_normal['variable'].isin(variables_candidatas_cc_normal)),
    'variable'])
if variables_tukey:
    tukey_df = posthoc_anova_tukey(cc_normal, variables_tukey, factor)
    
if variables_gh:
    gh_df = posthoc_welch_gameshowell(cc_normal, variables_gh, factor)
    
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(cc_normal, variables_dunn, factor)
if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
    
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
    
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para GH. Revisar


In [8]:
print(posthocs_gh)

{'dr_centrado_promedio':       A     B   mean(A)   mean(B)      diff        se         T        df  \
0    35  35-N  0.115451  0.009948  0.105503  0.046264  2.280447  5.116683   
1    35  35-R  0.115451  0.030667  0.084784  0.046541  1.821696  5.237805   
2  35-N  35-R  0.009948  0.030667 -0.020719  0.008677 -2.387919  6.728987   

       pval    hedges              variable  
0  0.147495  1.067370  dr_centrado_promedio  
1  0.251167  0.915825  dr_centrado_promedio  
2  0.109982 -1.347618  dr_centrado_promedio  }


In [15]:
pvalues_significativos_cc_normal

,variable,test,p_value,η²,ω²/ε²
2,dr_centrado_promedio,Welch's ANOVA,0.043381,0.407234,0.293921


## alta

In [9]:
resultados_cc_alta = df_resultados_cc[(df_resultados_cc['grupo'] == 'cc_alta')]
variables_resumen_por_test = (resultados_cc_alta[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("LOCOS ALTA - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_cc_alta.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_cc_alta = analizar_grupo(cc_alta, listas_por_test, factor)

pvalues_significativos_cc_alta = df_efectos_cc_alta.loc[(df_efectos_cc_alta['p_value'] < 0.05)]
variables_candidatas_cc_alta = pvalues_significativos_cc_alta['variable'].tolist()
print(pvalues_significativos_cc_alta)

LOCOS ALTA - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['r_centrado_promedio', 'r_centrado_mediana']
Puedo aplicar Kruskal-Wallis en: ['dr_centrado_promedio', 'dr_centrado_mediana']
              variable            test   p_value        η²     ω²/ε²
1  dr_centrado_mediana  Kruskal-Wallis  0.030365  0.427328  0.318247


In [18]:

variables_tukey = list(df_efectos_cc_alta.loc[
    (df_efectos_cc_alta['test'] == 'ANOVA') &
    (df_efectos_cc_alta['variable'].isin(variables_candidatas_cc_alta)),
    'variable'])

variables_gh = list(df_efectos_cc_alta.loc[
    (df_efectos_cc_alta['test'] == "Welch's ANOVA") &
    (df_efectos_cc_alta['variable'].isin(variables_candidatas_cc_alta)),
    'variable'])

variables_dunn = list(df_efectos_cc_alta.loc[
    (df_efectos_cc_alta['test'] == 'Kruskal-Wallis') &
    (df_efectos_cc_alta['variable'].isin(variables_candidatas_cc_alta)),
    'variable'])

if variables_tukey:
    tukey_df = posthoc_anova_tukey(cc_alta, variables_tukey, factor)
if variables_gh:
    gh_df = posthoc_welch_gameshowell(cc_alta, variables_gh, factor)
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(cc_alta, variables_dunn, factor)

if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")


Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [19]:
print(posthocs_dunn)

{'dr_centrado_mediana':    grupo1 grupo2   p-valor             variable
0      40     40  1.000000  dr_centrado_mediana
1      45     40  1.000000  dr_centrado_mediana
2      50     40  1.000000  dr_centrado_mediana
3      55     40  1.000000  dr_centrado_mediana
4      60     40  1.000000  dr_centrado_mediana
5      40     45  1.000000  dr_centrado_mediana
6      45     45  1.000000  dr_centrado_mediana
7      50     45  0.102117  dr_centrado_mediana
8      55     45  1.000000  dr_centrado_mediana
9      60     45  1.000000  dr_centrado_mediana
10     40     50  1.000000  dr_centrado_mediana
11     45     50  0.102117  dr_centrado_mediana
12     50     50  1.000000  dr_centrado_mediana
13     55     50  0.104212  dr_centrado_mediana
14     60     50  0.082423  dr_centrado_mediana
15     40     55  1.000000  dr_centrado_mediana
16     45     55  1.000000  dr_centrado_mediana
17     50     55  0.104212  dr_centrado_mediana
18     55     55  1.000000  dr_centrado_mediana
19     60     55

# ERIZOS

## baja

In [20]:
resultados_la_baja = df_resultados_la[(df_resultados_la['grupo'] == 'la_baja')]
variables_resumen_por_test = (resultados_la_baja[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS BAJA - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_la_baja.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_la_baja = analizar_grupo(la_baja, listas_por_test, factor)

pvalues_significativos_la_baja = df_efectos_la_baja.loc[(df_efectos_la_baja['p_value'] < 0.05)]
variables_candidatas_la_baja = pvalues_significativos_la_baja['variable'].tolist()
print(pvalues_significativos_la_baja)


ERIZOS BAJA - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['r_promedio', 'r_mediana']
Empty DataFrame
Columns: [variable, test, p_value, η², ω²/ε²]
Index: []


In [21]:

variables_tukey = list(df_efectos_la_baja.loc[
    (df_efectos_la_baja['test'] == 'ANOVA') &
    (df_efectos_la_baja['variable'].isin(variables_candidatas_la_baja)),
    'variable'])

variables_gh = list(df_efectos_la_baja.loc[
    (df_efectos_la_baja['test'] == "Welch's ANOVA") &
    (df_efectos_la_baja['variable'].isin(variables_candidatas_la_baja)),
    'variable'])

variables_dunn = list(df_efectos_la_baja.loc[
    (df_efectos_la_baja['test'] == 'Kruskal-Wallis') &
    (df_efectos_la_baja['variable'].isin(variables_candidatas_la_baja)),
    'variable'])

if variables_tukey:
    tukey_df = posthoc_anova_tukey(la_baja, variables_tukey, factor)
if variables_gh:
    gh_df = posthoc_welch_gameshowell(la_baja, variables_gh, factor)
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(la_baja, variables_dunn, factor)

if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")


Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [22]:
print(posthocs_dunn)

{}


## normal

In [26]:
resultados_la_normal = df_resultados_la[(df_resultados_la['grupo'] == 'la_normal')]
variables_resumen_por_test = (resultados_la_normal[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS NORMAL - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_la_normal.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_la_normal = analizar_grupo(la_normal, listas_por_test, factor)

pvalues_significativos_la_normal = df_efectos_la_normal.loc[(df_efectos_la_normal['p_value'] < 0.05)]
variables_candidatas_la_normal = pvalues_significativos_la_normal['variable'].tolist()
print(pvalues_significativos_la_normal)

ERIZOS NORMAL - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['r_promedio', 'r_mediana']
     variable   test   p_value        η²     ω²/ε²
0  r_promedio  ANOVA  0.001721  0.719994  0.645907
1   r_mediana  ANOVA  0.002309  0.703045  0.625092


In [27]:

variables_tukey = list(df_efectos_la_normal.loc[
    (df_efectos_la_normal['test'] == 'ANOVA') &
    (df_efectos_la_normal['variable'].isin(variables_candidatas_la_normal)),
    'variable'])

variables_gh = list(df_efectos_la_normal.loc[
    (df_efectos_la_normal['test'] == "Welch's ANOVA") &
    (df_efectos_la_normal['variable'].isin(variables_candidatas_la_normal)),
    'variable'])

variables_dunn = list(df_efectos_la_normal.loc[
    (df_efectos_la_normal['test'] == 'Kruskal-Wallis') &
    (df_efectos_la_normal['variable'].isin(variables_candidatas_la_normal)),
    'variable'])

if variables_tukey:
    tukey_df = posthoc_anova_tukey(la_normal, variables_tukey, factor)
if variables_gh:
    gh_df = posthoc_welch_gameshowell(la_normal, variables_gh, factor)
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(la_normal, variables_dunn, factor)

if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")


Hay df con post hoc para Tukey. Revisar
Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [28]:
print(posthocs_tukey)

{'r_promedio':   group1 group2  meandiff   p-adj   lower   upper  reject    variable
0     35   35-N   -1.3477  0.0481 -2.6836 -0.0118    True  r_promedio
1     35   35-R   -1.7831  0.0016 -2.7739 -0.7924    True  r_promedio
2   35-N   35-R   -0.4354  0.6691 -1.8043  0.9335   False  r_promedio, 'r_mediana':   group1 group2  meandiff   p-adj   lower   upper  reject   variable
3     35   35-N   -1.2473  0.0722 -2.6063  0.1117   False  r_mediana
4     35   35-R   -1.7534  0.0020 -2.7613 -0.7455    True  r_mediana
5   35-N   35-R   -0.5061  0.5955 -1.8987  0.8864   False  r_mediana}


## alta

In [ ]:
resultados_la_alta = df_resultados_la[(df_resultados_la['grupo'] == 'la_alta')]
variables_resumen_por_test = (resultados_la_alta[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS ALTA - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_la_alta.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
# df_efectos_la_alta = analizar_grupo(la_alta, listas_por_test, factor)

# pvalues_significativos_la_alta = df_efectos_la_alta.loc[(df_efectos_la_alta['p_value'] < 0.05)]
# variables_candidatas_la_alta = pvalues_significativos_la_alta['variable'].tolist()
# print(pvalues_significativos_la_alta)

ERIZOS ALTA - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar Welch's ANOVA en: ['r_promedio', 'r_mediana']


c:\Users\franc\anaconda3\envs\tesis\Lib\site-packages\statsmodels\regression\linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2


In [ ]:
# variables_tukey = list(df_efectos_la_alta.loc[
#     (df_efectos_la_alta['test'] == 'ANOVA') &
#     (df_efectos_la_alta['variable'].isin(variables_candidatas_la_alta)),
#     'variable'])

# variables_gh = list(df_efectos_la_alta.loc[
#     (df_efectos_la_alta['test'] == "Welch's ANOVA") &
#     (df_efectos_la_alta['variable'].isin(variables_candidatas_la_alta)),
#     'variable'])

# variables_dunn = list(df_efectos_la_alta.loc[
#     (df_efectos_la_alta['test'] == 'Kruskal-Wallis') &
#     (df_efectos_la_alta['variable'].isin(variables_candidatas_la_alta)),
#     'variable'])

# if variables_tukey:
#     tukey_df = posthoc_anova_tukey(la_alta, variables_tukey, factor)
# if variables_gh:
#     gh_df = posthoc_welch_gameshowell(la_alta, variables_gh, factor)
# if variables_dunn:
#     dunn_df = posthoc_kruskal_dunn(la_alta, variables_dunn, factor)

# if 'tukey_df' in locals():
#     posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
#     print("Hay df con post hoc para Tukey. Revisar")
# if 'gh_df' in locals():
#     posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
#     print("Hay df con post hoc para GH. Revisar")
# if 'dunn_df' in locals():
#     posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
#     print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para Tukey. Revisar
Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [ ]:
# print(posthocs_tukey)

{'a_mediana':   group1 group2  meandiff   p-adj   lower   upper  reject   variable
0     40     45   -0.0066  0.9991 -0.4659  0.4527   False  a_mediana
1     40     50   -0.4791  0.0283 -0.9043 -0.0538    True  a_mediana
2     45     50   -0.4725  0.0188 -0.8607 -0.0842    True  a_mediana, 'a_promedio':   group1 group2  meandiff   p-adj   lower   upper  reject    variable
3     40     45    0.0015  1.0000 -0.6568  0.6598   False  a_promedio
4     40     50   -0.6037  0.0522 -1.2132  0.0057   False  a_promedio
5     45     50   -0.6053  0.0337 -1.1617 -0.0489    True  a_promedio}
